# 01 — Ingest

Load raw data files into DuckDB and register source provenance.

**Data sources (all already downloaded to `data/raw/`):**
1. `wonder_mortality_2024.tsv` — Race × Sex × Cause (2024)
2. `wonder_mortality_ages_2024.tsv` — Race × Age × Cause (2024)
3. `wonder_mortality_gender_2024.tsv` — Sex × Age × Cause (2024)
4. `wonder_mortality_gender_2018_2024.tsv` — Sex × Race × Cause (2018–2024 aggregate)
5. `guttmacher_abortions.csv` — National abortion estimates + age breakdown + gestational age breakdown (2024)

All WONDER files were downloaded manually via the CDC WONDER query interface.
Guttmacher data compiled from published reports.


In [ ]:
import sys, os
from pathlib import Path

PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

import pandas as pd
import duckdb

from src.ingest import load_config
from src.clean_quality import (
    get_connection, load_to_duckdb, register_source, get_sources
)

cfg = load_config('config.yaml')
con = get_connection(cfg)
print(f'Project: {cfg["project_name"]}')
print(f'DuckDB: {cfg["settings"]["duckdb_file"]}')

## Helper: Parse WONDER TSV

WONDER tab-delimited exports have:
- A header row
- Data rows (Notes column is empty)
- A footer section with metadata (Notes column has text)

We strip the footer and handle special values ("Suppressed", "Not Applicable", "Unreliable").

In [ ]:
def parse_wonder_tsv(filepath: str | Path) -> pd.DataFrame:
    """Parse a CDC WONDER tab-delimited export file.
    
    Strips the footer/notes section, handles special values,
    and casts numeric columns.
    """
    df = pd.read_csv(
        filepath,
        sep='\t',
        na_values=['Not Applicable', 'Suppressed', 'Unreliable'],
        low_memory=False,
    )
    
    # Strip footer rows (Notes column has text in footer)
    df = df[df['Notes'].isna()].copy()
    df = df.drop(columns=['Notes'])
    
    # Cast Deaths and Population to numeric
    if 'Deaths' in df.columns:
        df['Deaths'] = pd.to_numeric(df['Deaths'], errors='coerce')
    if 'Population' in df.columns:
        df['Population'] = pd.to_numeric(df['Population'], errors='coerce')
    if 'Crude Rate' in df.columns:
        df['Crude Rate'] = pd.to_numeric(df['Crude Rate'], errors='coerce')
    
    # Normalize column names: lowercase, underscores
    df.columns = [
        c.strip().lower()
         .replace(' ', '_')
         .replace('-', '_')
         .replace('95%_confidence_interval', 'ci')
        for c in df.columns
    ]
    
    return df.reset_index(drop=True)

## 1. Load WONDER: Race × Sex × Cause (2024)

Primary file for national totals and sex-specific analysis.

In [ ]:
df_race_sex = parse_wonder_tsv('data/raw/wonder_mortality_2024.tsv')
print(f'Shape: {df_race_sex.shape}')
print(f'Columns: {list(df_race_sex.columns)}')
df_race_sex.head(3)

In [ ]:
load_to_duckdb(df_race_sex, 'mortality_race_sex', con)
register_source(
    con,
    table='mortality_race_sex',
    name='CDC WONDER Underlying Cause of Death, 2024',
    url='https://wonder.cdc.gov/ucd-icd10-expanded.html',
    license='Public domain (US government work)',
    notes='Single Race 6 × Sex × ICD-10 113 Cause List. Year 2024. Manual query export.',
    retrieved='2025-08-17',
)
print('✓ mortality_race_sex loaded')

## 2. Load WONDER: Race × Age × Cause (2024)

For age-stratified analysis aggregated across race.

In [ ]:
df_race_age = parse_wonder_tsv('data/raw/wonder_mortality_ages_2024.tsv')
print(f'Shape: {df_race_age.shape}')
print(f'Age groups: {sorted(df_race_age["five_year_age_groups"].unique().tolist())}')
df_race_age.head(3)

In [ ]:
load_to_duckdb(df_race_age, 'mortality_race_age', con)
register_source(
    con,
    table='mortality_race_age',
    name='CDC WONDER Underlying Cause of Death, 2024',
    url='https://wonder.cdc.gov/ucd-icd10-expanded.html',
    license='Public domain (US government work)',
    notes='Single Race 6 × Five-Year Age Groups × ICD-10 113 Cause List. Year 2024. Manual query export.',
    retrieved='2025-08-17',
)
print('✓ mortality_race_age loaded')

## 3. Load WONDER: Sex × Age × Cause (2024)

**Key file** for the female reproductive-age analysis. Direct sex-by-age breakdown
without needing to aggregate across race.

In [ ]:
df_sex_age = parse_wonder_tsv('data/raw/wonder_mortality_gender_2024.tsv')
print(f'Shape: {df_sex_age.shape}')
print(f'Sex values: {df_sex_age["sex"].unique().tolist()}')
print(f'Age groups: {sorted(df_sex_age["five_year_age_groups"].unique().tolist())}')
df_sex_age.head(3)

In [ ]:
load_to_duckdb(df_sex_age, 'mortality_sex_age', con)
register_source(
    con,
    table='mortality_sex_age',
    name='CDC WONDER Underlying Cause of Death, 2024',
    url='https://wonder.cdc.gov/ucd-icd10-expanded.html',
    license='Public domain (US government work)',
    notes='Sex × Five-Year Age Groups × ICD-10 113 Cause List. Year 2024. Primary file for female reproductive-age analysis.',
    retrieved='2025-08-17',
)
print('✓ mortality_sex_age loaded')

## 4. Load WONDER: Sex × Race × Cause (2018–2024 aggregate)

Multi-year reference for validating that 2024 patterns are consistent.

In [ ]:
df_sex_race_trend = parse_wonder_tsv('data/raw/wonder_mortality_gender_2018_2024.tsv')
print(f'Shape: {df_sex_race_trend.shape}')
df_sex_race_trend.head(3)

In [ ]:
load_to_duckdb(df_sex_race_trend, 'mortality_sex_race_trend', con)
register_source(
    con,
    table='mortality_sex_race_trend',
    name='CDC WONDER Underlying Cause of Death, 2018-2024',
    url='https://wonder.cdc.gov/ucd-icd10-expanded.html',
    license='Public domain (US government work)',
    notes='Sex × Single Race 6 × ICD-10 113 Cause List. Years 2018-2024 aggregated. Multi-year reference.',
    retrieved='2025-08-17',
)
print('✓ mortality_sex_race_trend loaded')

## 5. Load Guttmacher Abortion Estimates

National total (2024) and age-group breakdown derived from CDC Abortion Surveillance 2022 proportions.

In [ ]:
df_abortions = pd.read_csv('data/raw/guttmacher_abortions.csv')
print(f'Shape: {df_abortions.shape}')
df_abortions

In [ ]:
# Validate gestational age percentages sum to 100%
gestation_pct = df_abortions[
    (df_abortions['measure'] == 'gestation_pct')
    & (df_abortions['year'] == 2024)
]
gestation_sum = gestation_pct['value'].sum()
print(f'Gestational age distribution (2024):')
print(gestation_pct[['age_group', 'value', 'unit', 'notes']].to_string())
print(f'\nSum: {gestation_sum:.1f}% (should be 100%)')

# Show age distribution counts and totals
age_counts = df_abortions[df_abortions['measure'] == 'age_count']
print(f'\n\nAge-specific abortion counts (2024):')
print(age_counts[['age_group', 'value']].to_string())
print(f"Total (from age groups): {age_counts['value'].sum():,.0f}")

# Show gestational counts
gestation_counts = df_abortions[df_abortions['measure'] == 'gestation_count']
print(f'\n\nGestational age abortion counts (2024):')
print(gestation_counts[['age_group', 'value']].to_string())
print(f"Total (from gestation groups): {gestation_counts['value'].sum():,.0f}")


In [ ]:
load_to_duckdb(df_abortions, 'abortions', con)
register_source(
    con,
    table='abortions',
    name='Guttmacher Institute Monthly Abortion Provision Study, 2024',
    url='https://www.guttmacher.org/fact-sheet/induced-abortion-united-states',
    license='Published research (fair use for analysis)',
    notes='National total 1,124,000 for 2024. Age proportions and gestational age proportions from CDC Abortion Surveillance 2022. Rate: 16.7/1000 women 15-44. Gestational distribution: 78.6% ≤9w, 14.2% 10-13w, 6.1% 14-20w, 1.1% ≥21w (92.8% by week 13).',
    retrieved='2025-08-17',
)
print('✓ abortions loaded')


## 6. Load WONDER: Sex × Hispanic Origin × Race × Cause (2024)

Enables proper Non-Hispanic White / Non-Hispanic Black / Hispanic comparison
aligned with Guttmacher race categories.


In [ ]:
df_race_ethnicity = parse_wonder_tsv('data/raw/wonder_mortality_sex_race_ethnicity.tsv')
print(f'Shape: {df_race_ethnicity.shape}')
print(f'Hispanic Origin values: {df_race_ethnicity["hispanic_origin"].unique().tolist()}')
print(f'Single Race 6 values: {df_race_ethnicity["single_race_6"].unique().tolist()}')
df_race_ethnicity.head(3)


In [ ]:
load_to_duckdb(df_race_ethnicity, 'mortality_race_ethnicity', con)
register_source(
    con,
    table='mortality_race_ethnicity',
    name='CDC WONDER Underlying Cause of Death, 2024 (Sex × Hispanic Origin × Single Race 6)',
    url='https://wonder.cdc.gov/ucd-icd10-expanded.html',
    license='Public domain (US government work)',
    notes='Sex × Hispanic Origin × Single Race 6 × ICD-10 113 Cause List. Year 2024. '
          'Enables NH White / NH Black / Hispanic comparison aligned with Guttmacher.',
    retrieved='2026-08-22',
)
print('✓ mortality_race_ethnicity loaded')


## 6b. Load WONDER: Multi-Year Trend (1999-2020)

National mortality by cause and year for trend analysis.


In [ ]:
df_trend = parse_wonder_tsv('data/raw/Underlying Cause of Death, 1999-2020.tsv')
print(f'Shape: {df_trend.shape}')
load_to_duckdb(df_trend, 'mortality_trend_national', con)
register_source(con, table='mortality_trend_national',
    name='CDC WONDER Underlying Cause of Death, 1999-2020',
    url='https://wonder.cdc.gov/ucd-icd10.html',
    license='Public domain (US government work)',
    notes='Year × ICD-10 113 Cause List. 1999-2020. For trend analysis.',
    retrieved='2025-08-17')
print('✓ mortality_trend_national loaded')


## 6c. Load Guttmacher Historical Abortions

Year-by-year abortion counts for trend overlay.


In [ ]:
df_abort_hist = pd.read_csv('data/raw/guttmacher_abortions_historical.csv')
print(f'Shape: {df_abort_hist.shape}')
load_to_duckdb(df_abort_hist, 'abortions_trend', con)
register_source(con, table='abortions_trend',
    name='Guttmacher Institute Historical Abortion Estimates',
    url='https://www.guttmacher.org/fact-sheet/induced-abortion-united-states',
    license='Published research (fair use)',
    notes='Annual US abortion totals 1973-2024. For trend overlay with mortality data.',
    retrieved='2025-08-19')
print('✓ abortions_trend loaded')


## 6d. Load FBI SHR Homicide Data (2024)

Supplemental Homicide Reports for victim-offender race cross-tabulation.


In [ ]:
# FBI SHR - filter to 2024 for current year analysis
import io
df_shr_full = pd.read_csv('data/raw/SHR76_25.csv', low_memory=False)
print(f'Full SHR shape: {df_shr_full.shape}')

# Filter to 2024
df_shr_2024 = df_shr_full[df_shr_full['Year'] == 2024].copy()
print(f'SHR 2024: {len(df_shr_2024)} records')

load_to_duckdb(df_shr_2024, 'shr_homicides_2024', con)
register_source(con, table='shr_homicides_2024',
    name='FBI Supplemental Homicide Reports 2024',
    url='https://www.fbi.gov/how-we-can-help-you/more-fbi-services-and-information/ucr/supplementary-homicide-reports',
    license='Public domain (US government work)',
    notes='Victim-offender details for homicides. Filtered to 2024. For race cross-tabulation in 04-viz.',
    retrieved='2025-08-19')
print('✓ shr_homicides_2024 loaded')


## 7. Create Cause Display Names Mapping

Maps CDC ICD-10 113 Cause List names (stripped of # and ICD codes) to short
display names used in charts. This table is the single source of truth for
how cause names appear in all charts — no hardcoded mappings in notebooks.


In [ ]:
# Cause display names: CDC raw name -> short chart label
display_names_data = [
    ('Diseases of heart', 'Heart disease'),
    ('Malignant neoplasms', 'Cancer'),
    ('Accidents (unintentional injuries)', 'Accidents'),
    ('Chronic lower respiratory diseases', 'Respiratory disease'),
    ('Cerebrovascular diseases', 'Stroke'),
    ('Alzheimer disease', "Alzheimer's"),
    ('Diabetes mellitus', 'Diabetes'),
    ('Chronic liver disease and cirrhosis', 'Liver disease'),
    ('Intentional self-harm (suicide)', 'Suicide'),
    ('Nephritis, nephrotic syndrome and nephrosis', 'Kidney disease'),
    ('Assault (homicide)', 'Homicide'),
    ('Essential hypertension and hypertensive renal disease', 'Hypertension'),
    ('Parkinson disease', "Parkinson's"),
    ('Influenza and pneumonia', 'Flu & pneumonia'),
    ('Septicemia', 'Septicemia'),
    ('Human immunodeficiency virus (HIV) disease', 'HIV'),
    ('Certain conditions originating in the perinatal period', 'Perinatal conditions'),
    ('Congenital malformations, deformations and chromosomal abnormalities', 'Birth defects'),
    ('Pneumonitis due to solids and liquids', 'Aspiration pneumonia'),
    ('Aortic aneurysm and dissection', 'Aortic aneurysm'),
    ('In situ neoplasms, benign neoplasms and neoplasms of uncertain or unknown behavior', 'Other neoplasms'),
    ('Atherosclerosis', 'Atherosclerosis'),
    ('Nutritional deficiencies', 'Nutritional deficiencies'),
    ('Viral hepatitis', 'Viral hepatitis'),
    ('Peptic ulcer', 'Peptic ulcer'),
    ('Cholelithiasis and other disorders of gallbladder', 'Gallbladder disease'),
    ('Complications of medical and surgical care', 'Medical complications'),
    ('Pregnancy, childbirth and the puerperium', 'Pregnancy/childbirth'),
    ('Meningitis', 'Meningitis'),
    ('Anemias', 'Anemias'),
]

df_display = pd.DataFrame(display_names_data, columns=['cause_cdc', 'cause_display'])
load_to_duckdb(df_display, 'cause_display_names', con)
register_source(con, 'cause_display_names',
    'Display name mapping (internal)',
    notes='Maps CDC ICD-10 113 Cause names to short display names for charts. Single source of truth.')
print(f'✓ cause_display_names: {len(df_display)} mappings')
df_display.head(12)


## Verify: All tables and sources

In [ ]:
# Show all tables in the database
tables = con.execute("SHOW TABLES").df()
print(f'Tables in DuckDB ({len(tables)}):')
for t in tables['name'].tolist():
    count = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f'  {t:<30} {count:>8,} rows')

In [ ]:
# Show provenance metadata
get_sources(con)

## Quick sanity checks

In [ ]:
# Heart disease deaths (GR113-054) — the top single cause
heart_deaths = con.execute("""
    SELECT SUM(deaths) as total_deaths
    FROM mortality_race_sex
    WHERE icd_10_113_cause_list_code = 'GR113-054'
""").fetchone()[0]
print(f'Heart disease deaths (2024): {heart_deaths:,.0f}')
print(f'  Expected: ~680-700K')

# Cancer deaths (GR113-019)
cancer_deaths = con.execute("""
    SELECT SUM(deaths) as total_deaths
    FROM mortality_race_sex
    WHERE icd_10_113_cause_list_code = 'GR113-019'
""").fetchone()[0]
print(f'Cancer deaths (2024): {cancer_deaths:,.0f}')
print(f'  Expected: ~600-620K')

# Total US population (from race×sex, summing distinct race/sex combos)
total_pop = con.execute("""
    SELECT SUM(population) FROM (
        SELECT DISTINCT single_race_6, sex, population
        FROM mortality_race_sex
        WHERE icd_10_113_cause_list_code = 'GR113-054'
          AND population IS NOT NULL
          AND single_race_6 != 'Not Available'
    )
""").fetchone()[0]
print(f'Total US population (2024): {total_pop:,.0f}')

In [ ]:
# Total female population from the sex×age file
# (Population is the same regardless of cause — use any single cause as reference)
female_pop = con.execute("""
    SELECT SUM(population) as female_pop
    FROM mortality_sex_age
    WHERE sex = 'Female'
      AND icd_10_113_cause_list_code = 'GR113-054'
      AND five_year_age_groups NOT IN ('Not Stated')
      AND population IS NOT NULL
""").fetchone()[0]
print(f'Total female population (all ages, 2024): {female_pop:,.0f}')

# Female 15-44 population
repro_ages = ['15-19 years', '20-24 years', '25-29 years', '30-34 years', '35-39 years', '40-44 years']
repro_pop = con.execute(f"""
    SELECT SUM(population) as repro_pop
    FROM mortality_sex_age
    WHERE sex = 'Female'
      AND icd_10_113_cause_list_code = 'GR113-054'
      AND five_year_age_groups IN ({','.join(f"'{a}'" for a in repro_ages)})
      AND population IS NOT NULL
""").fetchone()[0]
print(f'Female population 15-44 (2024): {repro_pop:,.0f}')
print(f'  Expected: ~65-68 million')

In [ ]:
# Guttmacher total
abortion_total = con.execute("""
    SELECT value FROM abortions
    WHERE measure = 'national_total'
""").fetchone()[0]
print(f'\nGuttmacher 2024 national total: {abortion_total:,.0f}')
print(f'  vs Heart disease deaths: {heart_deaths:,.0f}')
print(f'  Abortion / Heart disease: {abortion_total/heart_deaths:.2f}x')
print(f'  vs Cancer deaths: {cancer_deaths:,.0f}')
print(f'  Abortion / Cancer: {abortion_total/cancer_deaths:.2f}x')

In [ ]:
con.close()
print('\n✓ Ingestion complete. Database saved to:', cfg['settings']['duckdb_file'])

---
**Next:** open `02-clean.ipynb` to filter, standardize, and build aggregated views.